# Linear Regression - Predicting Loan Amount

## Algorithm Background

**Linear Regression** is one of the most fundamental supervised machine learning algorithms used for predicting a continuous numerical outcome. It models the relationship between a dependent variable (target) and one or more independent variables (features) by fitting a linear equation to the observed data.

The model takes the form:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

Where:
- $y$ is the predicted value (LoanAmount)
- $\beta_0$ is the intercept (bias term)
- $\beta_1, \beta_2, ..., \beta_n$ are the coefficients (weights) for each feature
- $x_1, x_2, ..., x_n$ are the input features

The algorithm uses **Ordinary Least Squares (OLS)** to find the best-fitting line by minimizing the sum of squared differences between actual and predicted values.

## Why Linear Regression?

In this notebook, we predict **LoanAmount** - a continuous numerical variable representing the loan amount requested by applicants. Since the target is continuous, linear regression is a natural and interpretable choice as a baseline model. This complements the **Logistic Regression** notebook in this project, which predicts the binary **Loan_Status** (approved/rejected) on the same dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Dataset Description

**Dataset:** Loan Prediction Dataset  
**Source:** [Kaggle - Loan Prediction Dataset](https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset)  
**Size:** 614 rows x 13 columns

This dataset contains information about loan applicants and their loan details. Each record represents a loan application with the applicant's demographic and financial information.

| Attribute | Type | Description |
|-----------|------|-------------|
| Loan_ID | Object | Unique loan identifier |
| Gender | Object | Male / Female |
| Married | Object | Applicant married (Yes/No) |
| Dependents | Object | Number of dependents (0, 1, 2, 3+) |
| Education | Object | Graduate / Not Graduate |
| Self_Employed | Object | Self-employed (Yes/No) |
| ApplicantIncome | Integer | Applicant's income |
| CoapplicantIncome | Float | Co-applicant's income |
| LoanAmount | Float | Loan amount (in thousands) - **Target Variable** |
| Loan_Amount_Term | Float | Term of loan (in months) |
| Credit_History | Float | Credit history (1 = good, 0 = bad) |
| Property_Area | Object | Urban / Semiurban / Rural |
| Loan_Status | Object | Loan approved (Y/N) |

**Target Variable:** `LoanAmount` - the continuous loan amount requested by applicants, which we aim to predict using the remaining features.

In [ ]:
df = pd.read_csv("loan-dataset.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## Data Preprocessing

The dataset contains missing values in several columns. Our preprocessing strategy:
1. **Categorical columns** (Gender, Married, Dependents, Self_Employed): Fill with the **mode** (most frequent value)
2. **Numerical columns** (LoanAmount, Loan_Amount_Term, Credit_History): Fill with the **median** to reduce the impact of outliers
3. **Clean** the Dependents column by replacing '3+' with '3'
4. **Drop** `Loan_ID` (non-informative identifier) and `Loan_Status` (binary target used in the logistic regression notebook, not a meaningful predictor of loan amount)
5. **Encode** categorical variables using Label Encoding

In [ ]:
# Fill categorical columns with mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill numerical columns with median
for col in ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
# Clean Dependents column
df['Dependents'] = df['Dependents'].replace('3+', '3')

In [ ]:
# Drop Loan_ID (identifier) and Loan_Status (classification target, not relevant for regression)
df.drop(['Loan_ID', 'Loan_Status'], axis=1, inplace=True)

In [ ]:
# Label encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [ ]:
# Verify no missing values remain
df.isnull().sum()

## Exploratory Data Analysis

Before building the model, we visualize the target variable distribution and feature correlations to understand the data.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.histplot(df['LoanAmount'], kde=True)
plt.title('Distribution of LoanAmount')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['LoanAmount'])
plt.title('LoanAmount Boxplot')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Split Features and Target

In [ ]:
X = df.drop('LoanAmount', axis=1)
y = df['LoanAmount']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Feature Scaling

We apply StandardScaler to normalize the feature values. This ensures all features are on the same scale, which improves coefficient interpretability. The target variable (LoanAmount) is **not** scaled.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Train Model

We use `LinearRegression` from scikit-learn, which fits the model using Ordinary Least Squares (OLS) - minimizing the sum of squared residuals between actual and predicted values.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Intercept: {model.intercept_:.4f}")

In [ ]:
y_pred = model.predict(X_test)

## Results and Evaluation

We evaluate the model using standard regression metrics:
- **MSE (Mean Squared Error):** Average of squared differences between actual and predicted values
- **RMSE (Root Mean Squared Error):** Square root of MSE, in the same unit as the target
- **MAE (Mean Absolute Error):** Average of absolute differences, less sensitive to outliers
- **R-squared (R2):** Proportion of variance in the target explained by the model (1.0 = perfect fit, 0.0 = no better than predicting the mean)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE):  {rmse:.4f}")
print(f"Mean Absolute Error (MAE):       {mae:.4f}")
print(f"R-squared (R2):                  {r2:.4f}")

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual LoanAmount')
plt.ylabel('Predicted LoanAmount')
plt.title('Actual vs Predicted LoanAmount')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidths=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted LoanAmount')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients
feature_names = df.drop('LoanAmount', axis=1).columns
coefficients = pd.Series(model.coef_, index=feature_names).sort_values()

plt.figure(figsize=(8, 5))
coefficients.plot(kind='barh', color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Linear Regression Feature Coefficients')
plt.tight_layout()
plt.show()

## Critical Analysis and Discussion

### Model Performance
The R-squared value indicates what proportion of the variance in LoanAmount is explained by the model. A moderate R2 is expected given the limited number of features and the small dataset size (614 samples). The RMSE gives us the average prediction error in the same units as LoanAmount (thousands).

### Feature Importance
From the coefficient plot, we can observe which features most strongly influence the predicted loan amount. Features like **ApplicantIncome** and **CoapplicantIncome** are expected to have positive coefficients, as higher income applicants tend to request larger loans.

### Limitations
1. **Linearity assumption:** Linear regression assumes a linear relationship between features and target. Real-world loan amounts may have non-linear dependencies.
2. **Small dataset:** With only 614 samples, the model may not generalize well to unseen data.
3. **Skewed target:** LoanAmount is right-skewed with outliers, which can disproportionately affect OLS regression.
4. **Feature limitations:** The available features are mostly categorical with limited numerical predictors.

### Possible Improvements
- **Log transformation** of LoanAmount to reduce right-skewness and stabilize variance
- **Ridge or Lasso regression** to add regularization and reduce overfitting
- **Polynomial features** to capture non-linear relationships between features and target
- **Feature engineering** (e.g., creating a TotalIncome = ApplicantIncome + CoapplicantIncome feature)
- **Cross-validation** instead of a single train/test split for more robust evaluation
- **Outlier removal** for extreme LoanAmount values that may skew the model